# Treino da voz pt-BR do Fluwy Bot (GPU grátis)
Kaggle: *Settings › Accelerator › GPU T4/P100* e *Internet on*. Colab: *Ambiente › GPU*. Rode as células em ordem.
Base: gravações reais do CML-TTS (pt, CC BY 4.0), partindo da voz Faber (Piper). Cada avaliação usa Whisper (erros de pronúncia) e UTMOS (naturalidade).
No fim sai `piper2-nova-int8.zip`, pronto para o app (motor sherpa-onnx 1.10.46, com as correções de PAD e ditongos).

In [ ]:
%%bash
set -e
apt-get -qq update && apt-get -qq install -y build-essential cmake ninja-build espeak-ng ffmpeg >/dev/null
[ -d piper1 ] || git clone -q --depth 1 https://github.com/OHF-voice/piper1-gpl.git piper1
cd piper1 && pip -q install -e '.[train]' && pip -q install scikit-build cmake ninja && ./build_monotonic_align.sh >/dev/null && (python setup.py build_ext --inplace >/dev/null || true)
pip -q install "datasets<4" librosa soundfile scipy faster-whisper num2words onnx onnxruntime "sherpa-onnx==1.10.46" huggingface_hub
python -c "import torch;print('GPU:',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
%%writefile normalize_pt.py
"""Normalizador de texto pt-BR para TTS (espelho do que vai para o app em Kotlin)."""
import re

UN = "zero um dois três quatro cinco seis sete oito nove dez onze doze treze catorze quinze dezesseis dezessete dezoito dezenove".split()
DEZ = "_ _ vinte trinta quarenta cinquenta sessenta setenta oitenta noventa".split()
CEN = "_ cento duzentos trezentos quatrocentos quinhentos seiscentos setecentos oitocentos novecentos".split()

def _ate999(n, fem=False):
    if n == 100: return "cem"
    p = []
    c, r = divmod(n, 100)
    if c: p.append(CEN[c])
    if r:
        if r < 20: w = UN[r]
        else:
            d, u = divmod(r, 10); w = DEZ[d] + (" e " + UN[u] if u else "")
        p.append(w)
    s = " e ".join(p)
    if fem: s = re.sub(r"\bum\b", "uma", re.sub(r"\bdois\b", "duas", s)).replace("ntos", "ntas")
    return s

def num(n, fem=False):
    n = int(n)
    if n < 0: return "menos " + num(-n, fem)
    if n == 0: return "zero"
    parts = []
    for val, sg, pl in [(10**9, "um bilhão", "bilhões"), (10**6, "um milhão", "milhões"), (1000, "mil", "mil")]:
        q, n = divmod(n, val)
        if q: parts.append(sg if q == 1 else f"{_ate999(q) if q < 1000 else num(q)} {pl}")
    if n: parts.append(_ate999(n, fem))
    if len(parts) > 1 and (n < 100 or n % 100 == 0) and n: return ", ".join(parts[:-1]) + " e " + parts[-1]
    return " ".join(parts)

ORD = {1: "primeiro", 2: "segundo", 3: "terceiro", 4: "quarto", 5: "quinto", 6: "sexto", 7: "sétimo", 8: "oitavo", 9: "nono", 10: "décimo"}
MES = "janeiro fevereiro março abril maio junho julho agosto setembro outubro novembro dezembro".split()
ABBR = {"Sr.": "senhor", "Sra.": "senhora", "Dr.": "doutor", "Dra.": "doutora", "Av.": "avenida", "nº": "número", "n.º": "número",
        "km": "quilômetros", "kg": "quilos", "min": "minutos", "etc.": "etcétera", "obs.": "observação", "tel.": "telefone",
        "vc": "você", "pq": "porque", "tb": "também", "WhatsApp": "uótsap", "e-mails": "i-meils", "emails": "i-meils", "e-mail": "i-meil", "email": "i-meil", "OK": "oquei", "ok": "oquei"}

def _int(s): return int(s.replace(".", ""))

def normalize(t: str) -> str:
    t = re.sub(r"https?://\S+", "um link", t)
    t = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", lambda m: m.group(0).replace("@", " arroba ").replace(".", " ponto "), t)
    t = re.sub(r"[\U0001F300-\U0001FAFF☀-➿]", "", t)
    t = re.sub(r"[*_#`>]+", " ", t)
    def money(m):
        r, c = _int(m.group(1)), int((m.group(2) or "0").ljust(2, "0")[:2])
        s = f"{num(r)} {'real' if r == 1 else 'reais'}" if r else ""
        if c: s += (" e " if s else "") + f"{num(c)} {'centavo' if c == 1 else 'centavos'}"
        return s or "zero reais"
    t = re.sub(r"R\$\s?(\d{1,3}(?:\.\d{3})*|\d+)(?:,(\d{1,2}))?", money, t)
    t = re.sub(r"\b(\d{1,2})h(\d{2})?\b|\b(\d{1,2}):(\d{2})\b",
               lambda m: (lambda h, mi: f"{num(h, True)} {'hora' if h == 1 else 'horas'}" + (f" e {num(mi)}" if mi else ""))(
                   int(m.group(1) or m.group(3)), int(m.group(2) or m.group(4) or 0)), t)
    t = re.sub(r"\b(\d{1,2})/(\d{1,2})(?:/(\d{2,4}))?\b",
               lambda m: (f"{'primeiro' if int(m.group(1)) == 1 else num(m.group(1))} de {MES[int(m.group(2)) - 1]}"
                          + (f" de {num(int(m.group(3)) if len(m.group(3)) == 4 else 2000 + int(m.group(3)))}" if m.group(3) else ""))
               if 1 <= int(m.group(2)) <= 12 else m.group(0), t)
    t = re.sub(r"(\d+(?:,\d+)?)\s?%", lambda m: f"{_dec(m.group(1))} por cento", t)
    t = re.sub(r"\b(\d{1,2})[ºª°]", lambda m: ORD.get(int(m.group(1)), num(m.group(1))) if "ª" not in m.group(0) else ORD.get(int(m.group(1)), num(m.group(1)))[:-1] + "a", t)
    for k, v in ABBR.items(): t = re.sub(r"(?<!\w)" + re.escape(k) + r"(?!\w)", v, t)
    t = re.sub(r"\b\d{1,3}(?:\.\d{3})+(?:,\d+)?\b|\b\d+,\d+\b|\b\d+\b", lambda m: _dec(m.group(0)), t)
    FEM = r"(horas?|semanas?|pessoas?|vez|vezes|mensagens?|tarefas?|reuniões|reunião|notificações|notificação|páginas?|semanas?|mil pessoas)"
    t = re.sub(r"\bum " + FEM, r"uma \1", t); t = re.sub(r"\bdois " + FEM, r"duas \1", t)
    t = re.sub(r"\b(duzentos|trezentos|quatrocentos|quinhentos|seiscentos|setecentos|oitocentos|novecentos)(?=( e [\wê ]+)? " + FEM + ")", lambda m: m.group(1)[:-2] + "as", t)
    return re.sub(r"\s+", " ", t).strip()

def _dec(s):
    if "," in s:
        a, b = s.split(",", 1); return f"{num(_int(a))} vírgula {' '.join(UN[int(ch)] for ch in b) if len(b) > 2 else num(int(b))}"
    return num(_int(s))

if __name__ == "__main__":
    for s in ["Sua reunião é às 10h30 com o Dr. Paulo.", "O total ficou R$ 1.250,90 (25% de desconto) no dia 15/12/2026.",
              "Chegaram 3 e-mails; o 2º é do cliente: joao.silva@empresa.com.br", "Faltam 2 horas, 21 min e 1.000.000 de passos!"]:
        print(normalize(s))


In [ ]:
# Dados: locutor com mais horas no CML-TTS pt (ajuste MAX_HOURS: mais horas = voz melhor, treino mais longo)
import os; os.environ['MAX_HOURS']='3.0'
"""Prepara o conjunto de treino: escolhe no CML-TTS (pt, CC BY 4.0) a locutora/locutor com mais horas, filtra trechos
de 2–12 s, reamostra para 22050 Hz e grava wavs/ + metadata.csv (formato do Piper: arquivo|texto)."""
import os, sys, json, collections, re
import numpy as np, soundfile as sf
from datasets import load_dataset, Audio
import scipy.signal as ss

REPO = "freds0/cml_tts_dataset_portuguese"
OUT = "ds"; os.makedirs(f"{OUT}/wavs", exist_ok=True)
MAX_H = float(os.environ.get("MAX_HOURS", "2.0"))
ds = load_dataset(REPO, split="train", streaming=True)
first = next(iter(ds)); print("colunas:", {k: (type(v).__name__) for k, v in first.items() if k != "audio"}, flush=True)
spk_col = next(c for c in ("speaker_id", "speaker", "client_id", "spk_id", "spk") if c in first)
txt_col = next(c for c in ("transcript", "text", "transcription", "sentence", "normalized_text") if c in first)
gen_col = next((c for c in ("gender", "sex") if c in first), None)
print("speaker:", spk_col, "texto:", txt_col, "gênero:", gen_col, flush=True)

# 1ª passada (sem áudio): horas por locutor
dur = collections.Counter(); gender = {}
meta = load_dataset(REPO, split="train", streaming=True).remove_columns(["audio"]) if "duration" in first else None
it = meta if meta is not None else load_dataset(REPO, split="train", streaming=True)
for i, r in enumerate(it):
    d = r.get("duration") or (len(r["audio"]["array"]) / r["audio"]["sampling_rate"])
    dur[r[spk_col]] += d
    if gen_col: gender[r[spk_col]] = r[gen_col]
    if i > 60000: break
top = dur.most_common(12)
print("TOP", [(s, round(h / 3600, 2), gender.get(s)) for s, h in top], flush=True)
pref = os.environ.get("GENDER", "F").upper()[:1]
cands = [s for s, h in top if not gen_col or str(gender.get(s, "")).upper().startswith(pref)] or [top[0][0]]
spk = cands[0]; print("ESCOLHIDO", spk, round(dur[spk] / 3600, 2), "h", gender.get(spk), flush=True)

# 2ª passada: só esse locutor
rows = []; total = 0.0
for r in load_dataset(REPO, split="train", streaming=True):
    if r[spk_col] != spk: continue
    a = r["audio"]; x = np.asarray(a["array"], dtype=np.float32); sr = a["sampling_rate"]; d = len(x) / sr
    t = re.sub(r"\s+", " ", str(r[txt_col])).strip()
    lv = r.get("levenshtein")
    if lv is not None and lv < 0.95: continue   # transcrição conferida por um 2º reconhecedor (MMS): só trechos com texto confiável
    if not (2.0 <= d <= 12.0) or len(t) < 8: continue
    if sr != 22050: x = ss.resample_poly(x, 22050, sr).astype(np.float32)
    x = x / max(1e-6, float(np.abs(x).max())) * 0.9
    name = f"u{len(rows):05d}.wav"; sf.write(f"{OUT}/wavs/{name}", x, 22050, subtype="PCM_16")
    rows.append(f"{name}|{t}"); total += d
    if total >= MAX_H * 3600: break
open(f"{OUT}/metadata.csv", "w", encoding="utf-8").write("\n".join(rows) + "\n")
json.dump({"repo": REPO, "license": "CC BY 4.0", "speaker": str(spk), "gender": str(gender.get(spk)), "hours": round(total / 3600, 2), "clips": len(rows)},
          open(f"{OUT}/info.json", "w"), ensure_ascii=False)
print("PRONTO", len(rows), "trechos", round(total / 3600, 2), "h", flush=True)


In [ ]:
from huggingface_hub import hf_hub_download; import shutil
shutil.copy(hf_hub_download('rhasspy/piper-checkpoints','pt/pt_BR/faber/medium/epoch=1999-step=92000.ckpt',repo_type='dataset'),'start.ckpt'); print('ok')

## Treino (GPU)
`max_epochs` conta a partir de 2000 (o Faber já tem 1999). +400 épocas ≈ 2–4 h numa T4. Pode rodar de novo para continuar (usa o último checkpoint).

In [ ]:
%%bash
cd /kaggle/working 2>/dev/null || cd /content
CK=$(ls -t $(find runs -name '*.ckpt' 2>/dev/null) 2>/dev/null | head -1); [ -z "$CK" ] && CK=start.ckpt; echo "partindo de $CK"
python -m piper.train fit --data.voice_name nova --data.csv_path ds/metadata.csv --data.audio_dir ds/wavs \
  --model.sample_rate 22050 --data.espeak_voice pt-br --data.cache_dir cache --data.config_path config.json \
  --data.batch_size 32 --ckpt_path "$CK" --trainer.accelerator gpu --trainer.devices 1 \
  --trainer.max_epochs 2400 --trainer.max_time 00:08:00:00 --trainer.default_root_dir runs 2>&1 | grep -E "Epoch|error|Error" | tail -20

In [ ]:
%%bash
cd /kaggle/working 2>/dev/null || cd /content
CK=$(ls -t $(find runs -name '*.ckpt') | head -1); echo $CK
python -m piper.train.export_onnx --checkpoint "$CK" --output-file nova.onnx && cp config.json nova.onnx.json && ls -la nova.onnx

## Avaliação: Whisper (erros de pronúncia) + UTMOS (naturalidade)

In [ ]:
import re, numpy as np, soundfile as sf, torch
from piper import PiperVoice, SynthesisConfig
from faster_whisper import WhisperModel
from normalize_pt import normalize
SENT_PT = ["Olá! Tudo bem por aí? Eu sou o seu assistente.", "Estou aqui para ajudar com o que você precisar, é só falar.",
           "Prontinho, criei o fluxo que vai monitorar os seus e-mails a cada meia hora.", "Encontrei três mensagens importantes do seu cliente de São Paulo.",
           "A reunião com a equipe de vendas é amanhã às 10h30.", "O orçamento ficou em R$ 1.250,90, com 25% de desconto.",
           "Quer que eu prepare um rascunho da resposta para você revisar?", "Já liguei o lembrete para as quatro e meia da tarde.",
           "Não consegui acessar o site agora, vou tentar de novo em alguns minutos.", "Pode deixar, vou cuidar disso e te aviso quando terminar.",
           "O tempo amanhã deve ficar nublado, com chance de chuva no fim do dia.", "Achei o arquivo na pasta de documentos e já enviei para você.",
           "Bom dia! Você tem duas tarefas pendentes para hoje.", "O pagamento foi confirmado e o recibo já está no seu e-mail.",
           "Vou abrir o navegador e pesquisar os preços mais baratos.", "A bateria do celular está em vinte por cento, quer ativar a economia?",
           "Seu pedido chega na quinta-feira, entre nove e dezoito horas.", "Desculpe, não entendi. Pode repetir, por favor?",
           "Criei uma automação que responde os clientes no WhatsApp.", "O relatório de vendas do mês passado mostra um crescimento de doze por cento.",
           "Encontrei um voo para o Rio de Janeiro por quatrocentos reais.", "Lembre-se de tomar água e fazer uma pausa a cada hora.",
           "A música está tocando na sala, quer que eu aumente o volume?", "Tudo certo! Terminei a tarefa e salvei o resultado na pasta."]

v = PiperVoice.load('nova.onnx', config_path='nova.onnx.json', use_cuda=False); sr = v.config.sample_rate
asr = WhisperModel('medium', device='cuda' if torch.cuda.is_available() else 'cpu', compute_type='float16' if torch.cuda.is_available() else 'int8')
utmos = torch.hub.load('tarepan/SpeechMOS:v1.2.0', 'utmos22_strong', trust_repo=True).eval()
words = lambda s: re.findall(r'[a-záéíóúâêôãõçà]+', normalize(s).lower().replace('-', ' '))
def wer(r, h):
    r, h = words(r), words(h); d = list(range(len(h)+1))
    for i, rw in enumerate(r, 1):
        p, d[0] = d[0], i
        for j, hw in enumerate(h, 1): p, d[j] = d[j], min(d[j]+1, d[j-1]+1, p+(rw != hw))
    return d[len(h)], len(r)
E=T=0; mos=[]; allw=[]
for s in SENT_PT:
    a = np.concatenate([c.audio_float_array for c in v.synthesize(normalize(s), syn_config=SynthesisConfig(noise_scale=0.5, noise_w_scale=0.6))])
    sf.write('e.wav', np.concatenate([np.zeros(int(sr*.3), np.float32), a]), sr)
    hyp = ' '.join(x.text for x in asr.transcribe('e.wav', language='pt', beam_size=5)[0]); e, n = wer(s, hyp); E += e; T += n
    if e: print('ERRO:', s, ' >> ', hyp)
    with torch.no_grad(): mos.append(float(utmos(torch.from_numpy(a).float().unsqueeze(0), sr)))
    allw += [a, np.zeros(int(sr*.4), np.float32)]
sf.write('amostra.wav', np.concatenate(allw), sr)
print(f'Palavras erradas: {100*E/T:.2f}%  |  Naturalidade (UTMOS 1-5): {np.mean(mos):.2f}  (Faber original ~3.4)')
from IPython.display import Audio; Audio('amostra.wav')

## Pacote para o app
Gera `piper2-nova-int8.zip` (int8 + PAD após o início + ditongos + espeak do Piper). Baixe o arquivo e me envie no chat.

In [ ]:
import onnx, json, os, glob, shutil, zipfile, piper
from onnx import helper, TensorProto
from onnxruntime.quantization import quantize_dynamic, QuantType
cfg = json.load(open('nova.onnx.json')); tok = {k: v[0] for k, v in cfg['phoneme_id_map'].items()}
m = onnx.load('nova.onnx'); g = m.graph
meta = {'model_type':'vits','comment':'piper','language':'Portuguese','voice':cfg['espeak']['voice'],'version':'1','has_espeak':'1','has_g2pw':'0','n_speakers':str(cfg.get('num_speakers',1)),'sample_rate':str(cfg['audio']['sample_rate'])}
del m.metadata_props[:]
for k, v in meta.items(): p = m.metadata_props.add(); p.key = k; p.value = v
C = lambda n, dims, vals: helper.make_node('Constant', [], [n], value=helper.make_tensor('v', TensorProto.INT64, dims, vals))
nodes = [C('_s0',[1],[0]), C('_s1',[1],[1]), C('_s2',[1],[2]), C('_end',[1],[2**62]), C('_m1',[1],[-1]), C('_m2',[1],[-2]), C('_ax',[1],[1]), C('_flat',[1],[-1]),
         C('_pad',[1,1],[tok['_']]), C('_neg2',[1,2],[-1,-1]), C('_z1',[1,1],[0]), C('_z2',[1,2],[0,0]),
         helper.make_node('Slice',['input','_s0','_s1','_ax'],['_first']), helper.make_node('Slice',['input','_s1','_end','_ax'],['_rest']),
         helper.make_node('Concat',['_first','_pad','_rest'],['_x0'],axis=1)]
x = '_x0'
for k, (sym, did) in enumerate((s, i) for s, i in tok.items() if len(s) == 2 and s[0] in tok and s[1] in tok):
    p = f'_d{k}'; N = lambda op, ins, out, **at: (nodes.append(helper.make_node(op, ins, [p+out], **at)), p+out)[1]
    nodes += [C(p+'A',[1,1],[tok[sym[0]]]), C(p+'B',[1,1],[tok[sym[1]]]), C(p+'ID',[1,1],[did])]
    sh2 = N('Concat',[N('Slice',[x,'_s2','_end','_ax'],'t2'),'_neg2'],'sh2',axis=1)
    first = N('And',[N('Equal',[x,p+'A'],'ea'), N('Equal',[sh2,p+'B'],'eb')],'f'); rep = N('Where',[first,p+'ID',x],'rep'); fi = N('Cast',[first],'fi',to=7)
    d1 = N('Concat',['_z1',N('Slice',[fi,'_s0','_m1','_ax'],'h1')],'d1',axis=1); d2 = N('Concat',['_z2',N('Slice',[fi,'_s0','_m2','_ax'],'h2')],'d2',axis=1)
    keep = N('Reshape',[N('Equal',[N('Add',[d1,d2],'ds'),'_z1'],'k'),'_flat'],'k1'); x = N('Compress',[rep,keep],'x',axis=1)
nodes += [helper.make_node('Identity',[x],['_final']), helper.make_node('Shape',['_final'],['_shape']), helper.make_node('Slice',['_shape','_s1','_s2','_s0'],['_len'])]
for n in g.node:
    for i, s in enumerate(n.input):
        if s == 'input': n.input[i] = '_final'
        elif s == 'input_lengths': n.input[i] = '_len'
for n in reversed(nodes): g.node.insert(0, n)
d = 'piper2-nova-int8'; shutil.rmtree(d, True); os.makedirs(d)
onnx.save(m, 'tmp.onnx'); quantize_dynamic('tmp.onnx', f'{d}/model.onnx', weight_type=QuantType.QUInt8)
q = onnx.load(f'{d}/model.onnx'); del q.metadata_props[:]
for k, v in meta.items(): p = q.metadata_props.add(); p.key = k; p.value = v
onnx.save(q, f'{d}/model.onnx')
open(f'{d}/tokens.txt','w').write(''.join(f'{s} {i}\n' for s, i in tok.items() if len(s) == 1))
E = glob.glob(os.path.join(os.path.dirname(piper.__file__), '**', 'espeak-ng-data'), recursive=True)[0]
shutil.copytree(E, f'{d}/espeak-ng-data')
for f in glob.glob(f'{d}/espeak-ng-data/*_dict'):
    if os.path.basename(f) not in ('pt_dict', 'en_dict'): os.remove(f)
import sherpa_onnx
t = sherpa_onnx.OfflineTts(sherpa_onnx.OfflineTtsConfig(model=sherpa_onnx.OfflineTtsModelConfig(vits=sherpa_onnx.OfflineTtsVitsModelConfig(
    model=f'{d}/model.onnx', lexicon='', tokens=f'{d}/tokens.txt', data_dir=f'{d}/espeak-ng-data'), num_threads=2)))
a = t.generate(normalize('Olá! Tudo certo, já organizei sua agenda para amanhã.'), sid=0, speed=1.0); sf.write('teste_app.wav', a.samples, a.sample_rate)
with zipfile.ZipFile(f'{d}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for r, _, fs in os.walk(d):
        for f in fs: z.write(os.path.join(r, f), os.path.relpath(os.path.join(r, f), '.'))
print('Pacote:', f'{d}.zip', round(os.path.getsize(f'{d}.zip')/1e6, 1), 'MB  — teste no motor do app:', round(len(a.samples)/a.sample_rate, 1), 's')
Audio('teste_app.wav')